## Analysis (POS tags and sentence types)
Run the cells in order.

In [ ]:
%pip install nbformat
%run main_assignment.ipynb


   ---------------------------------------- 0/2 [fastjsonschema]
   ---------------------------------------- 0/2 [fastjsonschema]
   ---------------------------------------- 0/2 [fastjsonschema]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]

Note: you may need to restart the kernel to use updated packages.
98 entries loaded, no duplicates.
Multiword English entries: ['pounded yam', 'you all']


### Analysis

In [ ]:
import re

# The lexicon cells must already have been run (cell above does this with %run).
assert "LEXICON" in globals(), "Run main_assignment first (the cell above)."

# Rebuild the multiword list from LEXICON, so it is always in sync
MULTIWORD_KEYS = sorted((k for k in LEXICON if " " in k), key=len, reverse=True)

PUNCT = {"?", ".", "!", ",", ";"}

# ---------- Preprocessing ----------

CONTRACTIONS = {
    "isn't": ["is", "not"], "aren't": ["are", "not"], "wasn't": ["was", "not"],
    "weren't": ["were", "not"], "doesn't": ["does", "not"], "don't": ["do", "not"],
    "didn't": ["did", "not"], "won't": ["will", "not"], "hasn't": ["has", "not"],
    "haven't": ["have", "not"], "hadn't": ["had", "not"],
    "i'm": ["I", "am"], "you're": ["you", "are"], "we're": ["we", "are"],
    "they're": ["they", "are"], "he's": ["he", "is"], "she's": ["she", "is"],
    "it's": ["it", "is"], "i'll": ["I", "will"], "you'll": ["you", "will"],
    "he'll": ["he", "will"], "she'll": ["she", "will"], "we'll": ["we", "will"],
    "they'll": ["they", "will"], "i've": ["I", "have"], "you've": ["you", "have"],
    "we've": ["we", "have"], "they've": ["they", "have"],
}


def tokenize(sentence):
    sentence = sentence.replace("\u2019", "'").strip()
    return re.findall(r"\w+(?:'\w+)*|[?.!,;]", sentence)


def expand_contractions(words):
    out = []
    for w in words:
        lw = w.lower()
        if lw in CONTRACTIONS:
            out.extend(CONTRACTIONS[lw])
        elif lw.endswith("n't"):
            out.extend([w[:-3], "not"])
        elif lw.endswith("'s"):
            out.extend([w[:-2], "'s"])  # John's -> John 's (possessive)
        else:
            out.append(w)
    return out


# ---------- Word forms ----------

BE_FORMS = {"am", "is", "are", "was", "were", "be", "been"}
HAVE_FORMS = {"has", "have", "had"}
DO_FORMS = {"do", "does", "did"}
OBJ_PRONOUNS = {"me": "i", "him": "he", "them": "they", "us": "we"}

IRREGULAR_VERBS = {
    "ate": ("eat", "PAST"), "eaten": ("eat", "PART"),
    "went": ("go", "PAST"), "gone": ("go", "PART"),
    "came": ("come", "PAST"),
    "saw": ("see", "PAST"), "seen": ("see", "PART"),
    "heard": ("hear", "PAST"), "said": ("say", "PAST"),
    "wrote": ("write", "PAST"), "written": ("write", "PART"),
    "slept": ("sleep", "PAST"), "stood": ("stand", "PAST"), "sat": ("sit", "PAST"),
    "bought": ("buy", "PAST"), "sold": ("sell", "PAST"),
    "gave": ("give", "PAST"), "given": ("give", "PART"),
    "did": ("do", "PAST"), "done": ("do", "PART"),
    "knew": ("know", "PAST"), "known": ("know", "PART"),
    "taught": ("teach", "PAST"),
    "drank": ("drink", "PAST"), "drunk": ("drink", "PART"),
}
IRREGULAR_PLURALS = {"children": "child", "people": "person"}


def is_lexicon(word, pos):
    return word in LEXICON and LEXICON[word]["pos"] == pos


def verb_form(word):
    """Return (base, form) if word is a form of a lexicon verb, else None.
    form is one of BASE, ING, PAST, PART, 3SG."""
    w = word.lower()
    if is_lexicon(w, "V"):
        return (w, "BASE")
    if w in IRREGULAR_VERBS and is_lexicon(IRREGULAR_VERBS[w][0], "V"):
        return IRREGULAR_VERBS[w]
    cands = []
    if w.endswith("ing") and len(w) > 4:
        stem = w[:-3]
        cands += [(stem, "ING"), (stem + "e", "ING")]
        if len(stem) > 1 and stem[-1] == stem[-2]:
            cands.append((stem[:-1], "ING"))
    if w.endswith("ed") and len(w) > 3:
        stem = w[:-2]
        cands += [(stem, "PAST"), (w[:-1], "PAST")]
        if len(stem) > 1 and stem[-1] == stem[-2]:
            cands.append((stem[:-1], "PAST"))
        if w.endswith("ied"):
            cands.append((w[:-3] + "y", "PAST"))
    if w.endswith("ies"):
        cands.append((w[:-3] + "y", "3SG"))
    if w.endswith("es"):
        cands.append((w[:-2], "3SG"))
    if w.endswith("s"):
        cands.append((w[:-1], "3SG"))
    for base, form in cands:
        if is_lexicon(base, "V"):
            return (base, form)
    return None


def noun_plural(word):
    """Return the singular if word is a plural of a lexicon noun, else None."""
    w = word.lower()
    if w in IRREGULAR_PLURALS:
        return IRREGULAR_PLURALS[w]
    cands = []
    if w.endswith("ies"):
        cands.append(w[:-3] + "y")
    if w.endswith("es"):
        cands.append(w[:-2])
    if w.endswith("s"):
        cands.append(w[:-1])
    for base in cands:
        if is_lexicon(base, "N"):
            return base
    return None


def next_content_index(words, i):
    """Index of the next word after i, skipping 'not'."""
    while i < len(words) and words[i].lower() == "not":
        i += 1
    return i


# ---------- Lookup ----------

def lookup(word, pending_tense=None):
    lw = word.lower()

    if lw in OBJ_PRONOUNS:
        return {"en": word, "base": OBJ_PRONOUNS[lw], "role": "obj",
                **LEXICON[OBJ_PRONOUNS[lw]]}

    if lw == "'s":
        return {"en": word, "pos": "POSS_S", "yo": None,
                "note": "John's book -> ìwé John (owner goes after the thing)"}

    if lw in LEXICON:
        tok = {"en": word, "base": lw, **LEXICON[lw]}
        if tok["pos"] == "V":
            tok["form"] = "BASE"
            if pending_tense:
                tok["tense"] = pending_tense
        return tok

    vf = verb_form(lw)
    if vf:
        base, form = vf
        tok = {"en": word, "base": base, "pos": "V",
               "yo": LEXICON[base]["yo"], "form": form}
        if form == "PAST" or pending_tense == "PAST":
            tok["tense"] = "PAST"
        return tok

    singular = noun_plural(lw)
    if singular:
        return {"en": word, "base": singular, "pos": "N",
                "yo": LEXICON[singular]["yo"], "number": "PL"}

    if word[0].isupper():
        return {"en": word, "pos": "PN", "yo": word}  # proper noun, passed through
    return {"en": word, "pos": "UNK", "yo": None, "note": "not in lexicon"}


# ---------- Analysis ----------

def analyze(sentence):
    words = expand_contractions(tokenize(sentence))
    n = len(words)
    tokens = []
    pending_tense = None
    i = 0

    while i < n:
        w = words[i]
        lw = w.lower()

        if lw in PUNCT:
            tokens.append({"en": w, "pos": "PUNCT", "yo": None})
            i += 1
            continue

        # Multiword entries first ("pounded yam", "you all")
        matched = False
        for key in MULTIWORD_KEYS:
            parts = key.split()
            if [x.lower() for x in words[i:i + len(parts)]] == parts:
                tokens.append({"en": " ".join(words[i:i + len(parts)]), "base": key,
                               **LEXICON[key]})
                i += len(parts)
                matched = True
                break
        if matched:
            continue

        # Auxiliaries become aspect markers (or are dropped)
        if lw == "will":
            tokens.append({"en": w, "pos": "ASP", "tag": "FUT", "yo": None})
            i += 1
            continue

        if lw in BE_FORMS:
            j = next_content_index(words, i + 1)
            vf = verb_form(words[j]) if j < n else None
            if vf and vf[1] == "ING":
                tokens.append({"en": w, "pos": "ASP", "tag": "PROG", "yo": None,
                               "tense": "PAST" if lw in ("was", "were") else "PRES"})
            else:
                tokens.append({"en": w, "pos": "COP", "yo": None,
                               "note": "be as main verb; the rules decide ni / jẹ́ / wà"})
            i += 1
            continue

        if lw in HAVE_FORMS:
            j = next_content_index(words, i + 1)
            vf = verb_form(words[j]) if j < n else None
            if vf and vf[1] in ("PAST", "PART"):
                tokens.append({"en": w, "pos": "ASP", "tag": "PERF", "yo": None})
                i += 1
                continue

        if lw in DO_FORMS:
            nxt = words[i + 1].lower() if i + 1 < n else ""
            if nxt == "not" or (i == 0 and words[-1] == "?"):
                if lw == "did":
                    pending_tense = "PAST"
                i += 1  # do-support has no Yorùbá equivalent, so drop it
                continue

        tokens.append(lookup(w, pending_tense))
        if tokens[-1]["pos"] == "V":
            pending_tense = None
        i += 1

    # Pronoun role: before the verb = subject, after = object.
    # A new clause (and / but / because / who / comma) starts the count again.
    seen_verb = False
    for tok in tokens:
        if tok["pos"] in ("CONJ", "SUB", "REL") or (
                tok["pos"] == "PUNCT" and tok["en"] == ","):
            seen_verb = False
        if tok["pos"] == "V":
            seen_verb = True
        if tok["pos"] == "PRON":
            role = tok.get("role") or ("obj" if seen_verb else "subj")
            tok["role"] = role
            tok["yo"] = tok[role]
            if tok["yo"] is None:
                tok["note"] = "3sg object copies the verb's final vowel"

    return tokens


def show(tokens):
    parts = []
    for t in tokens:
        label = t["pos"] + ("." + t["tag"] if "tag" in t else "")
        if "form" in t and t["form"] != "BASE":
            label += "." + t["form"]
        if t.get("number"):
            label += "." + t["number"]
        parts.append(f"{t['en']}[{label}]" + (f"={t['yo']}" if t.get("yo") else ""))
    return "  ".join(parts)


### Compound and complex sentences

In [ ]:
import unicodedata

# ---------- Extra words for compound and complex sentences ----------
# Added here so main_assignment stays untouched. REVIEW every Yorùbá form.
EXTRA_ENTRIES = [
    ("or", "CONJ", "tàbí"),
    ("that", "REL", "tí"),
    ("who", "REL", "tí"),
    ("which", "REL", "tí"),
    ("when", "SUB", "nígbà tí"),
    ("if", "SUB", "bí"),
    ("although", "SUB", "bí ó tilẹ̀ jẹ́ pé"),
    ("after", "SUB", "lẹ́yìn tí"),
]
for english, pos, yoruba in EXTRA_ENTRIES:
    LEXICON.setdefault(english, {"pos": pos, "yo": unicodedata.normalize("NFC", yoruba)})

# "because" starts a dependent clause, so it is tagged SUB, not CONJ
LEXICON["because"]["pos"] = "SUB"

# Tags: CONJ = and / but / or (joins equal clauses)
#       SUB  = because / when / if (starts a dependent clause)
#       REL  = who / which / that (relative clause)

SUBJECT_LIKE = {"PRON", "PN", "N", "DET", "POSS"}
VERBAL = {"V", "COP", "ASP"}


def clause_info(tokens):
    """Classify a tagged sentence as simple, compound, complex or compound-complex."""
    # Dependent clauses = subordinators + relative markers
    n_dep = sum(1 for t in tokens if t["pos"] in ("SUB", "REL"))

    # Each CONJ that is followed by a new subject + verb starts another independent clause
    n_new = 0
    for i, t in enumerate(tokens):
        if t["pos"] != "CONJ":
            continue
        seg = []
        j = i + 1
        while j < len(tokens) and tokens[j]["pos"] != "CONJ" and not (
                tokens[j]["pos"] == "PUNCT" and tokens[j]["en"] in ".?!"):
            seg.append(tokens[j]["pos"])
            j += 1
        for k, pos in enumerate(seg):
            if pos in SUBJECT_LIKE and any(p in VERBAL for p in seg[k + 1:]):
                n_new += 1
                break

    n_indep = 1 + n_new
    if n_indep >= 2 and n_dep >= 1:
        kind = "compound-complex"
    elif n_indep >= 2:
        kind = "compound"
    elif n_dep >= 1:
        kind = "complex"
    else:
        kind = "simple"
    return {"type": kind, "independent": n_indep, "dependent": n_dep}


def describe(sentence):
    tokens = analyze(sentence)
    info = clause_info(tokens)
    print(sentence)
    print(f"   Type: {info['type']} "
          f"({info['independent']} independent, {info['dependent']} dependent)")
    print("  ", show(tokens), "\n")


### Test

In [ ]:
simple = [
    "He is eating rice",
    "I don't want water",
    "My father bought a big car",
    "Did you see the child?",
]
compound = [
    "I eat rice and he drinks water",
    "My father bought a car but my mother bought a book",
    "She will go to the market or he will go to school",
]
complex_ = [
    "The child who ate rice is big",
    "She went to the market because she wanted bread",
    "When he came, I was eating rice",
]
compound_complex = [
    "My father bought a car because he wanted a big house, but my mother bought a book",
    "The teacher who taught my child went to the market and she bought fish because she wanted food",
    "When he came, I was eating rice but she was sleeping",
]
for group in (simple, compound, complex_, compound_complex):
    for s in group:
        describe(s)

He is eating rice
   Type: simple (1 independent, 0 dependent)
   He[PRON]=ó  is[ASP.PROG]  eating[V.ING]=jẹ  rice[N]=ìrẹsì 

I don't want water
   Type: simple (1 independent, 0 dependent)
   I[PRON]=mo  not[NEG]=kò  want[V]=fẹ́  water[N]=omi 

My father bought a big car
   Type: simple (1 independent, 0 dependent)
   My[POSS]=mi  father[N]=bàbá  bought[V.PAST]=ra  a[DET]=kan  big[ADJ]=ńlá  car[N]=ọkọ̀ ayọ́kẹ́lẹ́ 

Did you see the child?
   Type: simple (1 independent, 0 dependent)
   you[PRON]=o  see[V]=rí  the[DET]=náà  child[N]=ọmọ  ?[PUNCT] 

I eat rice and he drinks water
   Type: compound (2 independent, 0 dependent)
   I[PRON]=mo  eat[V]=jẹ  rice[N]=ìrẹsì  and[CONJ]=àti  he[PRON]=ó  drinks[V.3SG]=mu  water[N]=omi 

My father bought a car but my mother bought a book
   Type: compound (2 independent, 0 dependent)
   My[POSS]=mi  father[N]=bàbá  bought[V.PAST]=ra  a[DET]=kan  car[N]=ọkọ̀ ayọ́kẹ́lẹ́  but[CONJ]=ṣùgbọ́n  my[POSS]=mi  mother[N]=ìyá  bought[V.PAST]=ra  a[DET]=kan  book